In [5]:
import argparse
import pdb
import os
import math
import sys
from timeit import default_timer as timer

import numpy as np
import pandas as pd

### PyTorch Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler, WeightedRandomSampler, RandomSampler, SequentialSampler, sampler

import warnings
warnings.filterwarnings("ignore")

import pickle
import re

import h5py
from scipy import stats
from sklearn.preprocessing import StandardScaler

#### Newly defined library
from datasets.dataset_pfs_pnu import *
from models.model_double import ABMIL
from utils.focal_loss import FocalLoss, calculate_class_weights

from tensorboardX import SummaryWriter
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from tqdm import tqdm

os.environ["NCCL_DEBUG"] = "INFO"
os.environ["NCCL_IB_DISABLE"] = "1"  
os.environ["NCCL_P2P_DISABLE"] = "1"  
os.environ["NCCL_ASYNC_ERROR_HANDLING"] = "1"
import random

def even_sample(data, n_samples):
    n = len(data)
    indices = np.linspace(0, n-1, n_samples, dtype=int)
    return [data[i] for i in indices]

def even_sample_random(data, n_samples, seed=None):
    """균등하면서도 랜덤한 샘플링 (층화 샘플링)"""
    if seed is not None:
        np.random.seed(seed)
    
    n = len(data)
    if n_samples >= n:
        return data.copy()
    
    # 데이터를 n_samples개의 구간으로 나누기
    strata_size = n / n_samples
    
    indices = []
    for i in range(n_samples):
        # 각 구간의 시작과 끝
        start = int(i * strata_size)
        end = int((i + 1) * strata_size)
        if i == n_samples - 1:  # 마지막 구간
            end = n
        
        # 해당 구간에서 랜덤하게 하나 선택
        idx = np.random.randint(start, end)
        indices.append(idx)
    
    return [data[i] for i in sorted(indices)]

class Dataset_PFS_PNU(Dataset) : 
    def __init__(self, csv_path = '', csv_path2='',
                 data_dir = '', 
                 shuffle = False, seed = 7, label_col = 'label', sampling_cnt = None, train=True) : 
        
        self.seed = seed
        self.csv_path = csv_path
        self.csv_path2 = csv_path2
        self.data_dir = data_dir
        self.label_col = label_col
        self.train = train
        self.sampling_cnt = sampling_cnt

        slide_data = pd.read_csv(csv_path)
        feat_names = self.feat_naming(slide_data)
        
        if shuffle:
            np.random.seed(seed)
            np.random.shuffle(slide_data)
            np.random.shuffle(feat_names)
        
        self.slide_data = slide_data
        self.feat_names = feat_names
         
    def feat_naming(self, slide_data) : 
        feat_names = []

        for name in os.listdir(self.data_dir) : 
            if name.split('.')[-1] != 'done' :        
                try : 
                    patientno = float(name.split('-')[1])
                    if patientno in list(slide_data['Patient_no']) : 
                        feat_names.append(patientno)
                except : 
                    patientno = float(name.split('-')[1][:-3])
                    if patientno in list(slide_data['Patient_no']) : 
                        feat_names.append(patientno)
                
        return list(set(feat_names))
    
    def slide_sector_agg(self, patientno) : 
        slide_sector = []
        for word in os.listdir(self.data_dir) : 
            if word.split('.')[-1] != 'done' : 
                try : 
                    if float(word.split('-')[1]) == patientno : 
                        slide_sector.append(word)
                    else : 
                        continue
                except : 
                    if float(word.split('-')[1][:-3]) == patientno : 
                        slide_sector.append(word)
                    else : 
                        continue
        
        return slide_sector
    
    
    def __getitem__(self, idx) : 
        
        feat_file_name = self.feat_names[idx]
        feat_name = feat_file_name
        patientno = feat_name
        label = float(self.slide_data[self.slide_data['Patient_no'] == patientno][self.label_col])
        
        slide_sector = sorted(self.slide_sector_agg(patientno))
        length = len(slide_sector)
        
        try : 
            slide_sector = even_sample_random(slide_sector, self.sampling_cnt, self.seed)
        except : 
            pass
        
        path_features = []
        for feature_name in slide_sector : 
            with h5py.File(os.path.join(self.data_dir, feature_name), 'r') as file : 
                feature = file['features'][:]
            path_features.append(feature)
        
        path_features = torch.tensor(np.vstack(path_features))
        return (path_features, label, feat_name, slide_sector)
        
    def __len__(self) : 
        return len(self.feat_names)



def get_split_loader(dataset, training = False, batch_size=1):
    """
        return either the validation loader or training loader 
    """
    kwargs = {'num_workers': 4} 
    
    if training : 
        loader = DataLoader(dataset, batch_size=batch_size, sampler = RandomSampler(dataset), **kwargs)   
    else : 
        loader = DataLoader(dataset, batch_size=batch_size, sampler = SequentialSampler(dataset),  **kwargs)
    return loader

def l1_reg_all(model, reg_type=None):
    l1_reg = None
    for W in model.parameters():
        if l1_reg is None:
            l1_reg = torch.abs(W).sum()
        else:
            l1_reg = l1_reg + torch.abs(W).sum()
    return l1_reg

def l2_reg_all(model):
    l2_reg = None
    for W in model.parameters():
        if l2_reg is None:
            l2_reg = torch.sum(W ** 2)
        else:
            l2_reg = l2_reg + torch.sum(W ** 2)
    return l2_reg


def enable_dropout(model):
    """Enable all Dropout layers in the model"""
    for m in model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train()

def mc_dropout_inference(model, data_wsi, n_samples=10):
    """Inference by MC Dropout
    
    Args:
        model: model
        data_wsi: Input data
        n_samples: # of sampling
    
    Returns:
        mean logits, mean prob, pred val, attention scores, uncertainty
    """
    model.eval()
    enable_dropout(model)  # Dropout activation
    
    all_logits = []
    all_probs = []
    all_attention_scores = []
    
    with torch.no_grad():
        for _ in range(n_samples):
            binary_logits, binary_probs, _, attention_scores = model(x=data_wsi)
            all_logits.append(binary_logits)
            all_probs.append(binary_probs)
            all_attention_scores.append(attention_scores)
    
    # calculate mean
    mean_logits = torch.stack(all_logits).mean(dim=0)
    mean_probs = torch.stack(all_probs).mean(dim=0)
    mean_pred = torch.argmax(mean_logits, dim=1)
    
    # calculate uncertainty (variance of probability)
    prob_std = torch.stack(all_probs).std(dim=0)
    uncertainty = prob_std.mean().item()
    
    # mean of attention scores 
    mean_attention_scores = {}
    for key in all_attention_scores[0].keys():
        mean_attention_scores[key] = torch.stack([att[key] for att in all_attention_scores]).mean(dim=0)
    
    return mean_logits, mean_probs, mean_pred, mean_attention_scores, uncertainty


In [6]:
### Training settingst
parser = argparse.ArgumentParser(description='Configurations for Survival Analysis on TCGA Data.')
### Checkpoint + Misc. Pathing Parameters
parser.add_argument('--gamma',   type=float, default=1.0, help='power of focal loss penalty to majority')
parser.add_argument('--dropout',   type=float, default=0.25, help='Dropout ratio') ######## majority 클래스에 fit되는 것을 방지하기 위해 적은 값을 사용
parser.add_argument('--attn_branch',   type=int, default=2, help='# of attention branches') 
parser.add_argument('--mc_dropout', action='store_true', default=False, help='Use or not with test mc dropout')
parser.add_argument('--mc_samples',   type=int, default=10, help='# of samples for MC dropouts')
parser.add_argument('--lambda_reg',   type=float, default=5e-7, help='regularized term weights')
parser.add_argument('--epoch',   type=int, default=100, help='# of epochs')
parser.add_argument('--gc',   type=int, default=16, help='gradient accumulation')
parser.add_argument('--lr',   type=float, default=5e-6, help='learning rate')
parser.add_argument('--wd',   type=float, default=5e-7, help='weight decay')
parser.add_argument('--layer_norm', action='store_true', default=False, help='Use layer normalization for aggregation features')
parser.add_argument('--model_large', action='store_true', default=False, help='Use larger model capacity')
parser.add_argument('--seed', 	type=int, default=7, help='Random seed for reproducible experiment (default: 1)')

args = parser.parse_args(args=[])

In [7]:
model1 = ABMIL(n_classes=2, mode='binary', dropout=args.dropout, layer_norm = args.layer_norm, attention_branch=args.attn_branch)
model2 = ABMIL(n_classes=2, mode='binary', dropout=args.dropout, layer_norm = args.layer_norm, attention_branch=args.attn_branch)
model3 = ABMIL(n_classes=2, mode='binary', dropout=args.dropout, layer_norm = args.layer_norm, attention_branch=args.attn_branch)
model4 = ABMIL(n_classes=2, mode='binary', dropout=args.dropout, layer_norm = args.layer_norm, attention_branch=args.attn_branch)
model5 = ABMIL(n_classes=2, mode='binary', dropout=args.dropout, layer_norm = args.layer_norm, attention_branch=args.attn_branch)

model1.load_state_dict(torch.load('/mnt/fileserver/Pathology/PNU/5_fold_double_weights/1_fold_double_weight.pth'))
model2.load_state_dict(torch.load('/mnt/fileserver/Pathology/PNU/5_fold_double_weights/2_fold_double_weight.pth'))
model3.load_state_dict(torch.load('/mnt/fileserver/Pathology/PNU/5_fold_double_weights/3_fold_double_weight.pth'))
model4.load_state_dict(torch.load('/mnt/fileserver/Pathology/PNU/5_fold_double_weights/4_fold_double_weight.pth'))
model5.load_state_dict(torch.load('/mnt/fileserver/Pathology/PNU/5_fold_double_weights/5_fold_double_weight.pth'))

args.gpu = 0
device = "cuda:{}".format(args.gpu)
model1 = model1.to(device)
model2 = model2.to(device)
model3 = model3.to(device)
model4 = model4.to(device)
model5 = model5.to(device)

binary_loss = nn.CrossEntropyLoss()

model1.eval()
model2.eval()
model3.eval()
model4.eval()
model5.eval()

ABMIL(
  (wsi_net): Sequential(
    (0): Linear(in_features=1024, out_features=1024, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (instancenorm): InstanceNorm1d(1024, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
  (path_attention_head): Attn_Net_Gated(
    (attention_a): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): Tanh()
    )
    (attention_b): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): Sigmoid()
    )
    (attention_c): Linear(in_features=1024, out_features=2, bias=True)
  )
  (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (path_rho): Sequential(
    (0): Linear(in_features=1024, out_features=1024, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (classifier): Linear(in_features=1024, out_features=2, bias=True)
  (binary_classifier): Linear(in_features=1024, out_features=2, bias=True)
)

In [8]:
train_alls = []

for fold in range(5) : 

    args.train_clinical = '/home/yscho/MIL_train/folds_2year/val_fold{}.csv'.format(fold)
    args.train_img = '/home/yscho/code_cloud_MIL_train/output_patch_sampling/train_sampled_pct01p00' ## 1% patches from developing set

    sampling_cnt = 19

    train_dataset = Dataset_PFS_PNU(csv_path = args.train_clinical,
                        data_dir = args.train_img,
                        label_col = 'BCR', sampling_cnt=sampling_cnt, seed=args.seed) ## 샘플 고정

    train_loader = get_split_loader(train_dataset, training=False, batch_size=1)

    train_epoch_probs = []
    train_names = []
    train_labels = []
    train_epoch_targets = []

    for batch_idx, (data_WSI, c, name, _) in enumerate(tqdm(train_loader)):
        data_WSI = data_WSI.to(device)
        c = c.type(torch.LongTensor).to(device)
        train_names.append(int(name.item()))
        train_labels.append(c.item())
        
        with torch.no_grad() : 
            _, binary_probs, _, attention_scores = model1(x=data_WSI)
            prob_class = binary_probs[0, 1].cpu().item()  # class 1의 확률  
            train_epoch_probs.append(prob_class)  # class 1 확률만 저장 (AUC용)
            target = c.cpu().item()
            train_epoch_targets.append(target)

    train_epoch_probs = np.array(train_epoch_probs)
    auc_score = roc_auc_score(train_epoch_targets, train_epoch_probs)

    df_risk_score_train = pd.DataFrame({"Patient_no":train_names, "risk_scores":train_epoch_probs, "BCR": train_labels})
    train_alls.append(df_risk_score_train)

100%|██████████| 108/108 [00:34<00:00,  3.13it/s]


In [ ]:
df_risk_score_train_fold = pd.concat([train_alls[0], train_alls[1], 
                                      train_alls[2], train_alls[3], 
                                      train_alls[4]])
df_risk_score_train_fold.to_csv("TRAIN_RISK_SCORES_FINAL_PNU_1patch_19slide.csv")

In [9]:
args.test_clinical = '/home/yscho/MIL_train/folds_2year/test.csv'
args.test_img = '/mnt/fileserver/Pathology/PNU/test_sampled_pct01p00' ## 1% patches from test set

sampling_cnt = 19

test_dataset = Dataset_PFS_PNU(csv_path = args.test_clinical,
                    data_dir = args.test_img,
                    label_col = 'BCR', sampling_cnt=sampling_cnt, seed=args.seed) ## 샘플 고정

test_loader = get_split_loader(test_dataset, training=False, batch_size=1)

In [10]:
test_epoch_probs = []
test_names = []
test_labels = []
test_epoch_targets = []

for batch_idx, (data_WSI, c, name, _) in enumerate(tqdm(test_loader)):
    data_WSI = data_WSI.to(device)
    c = c.type(torch.LongTensor).to(device)
    test_names.append(int(name.item()))
    test_labels.append(c.item())
    
    with torch.no_grad() : 
        _, binary_probs1, _, _ = model1(x=data_WSI)
        _, binary_probs2, _, _ = model2(x=data_WSI)
        _, binary_probs3, _, _ = model3(x=data_WSI)
        _, binary_probs4, _, _ = model4(x=data_WSI)
        _, binary_probs5, _, _ = model5(x=data_WSI)
    
    probs = []
    with torch.no_grad():
        prob_class1 = binary_probs1[0, 1].cpu().item()  # class 1의 확률
        prob_class2 = binary_probs2[0, 1].cpu().item()  # class 1의 확률
        prob_class3 = binary_probs3[0, 1].cpu().item()  # class 1의 확률
        prob_class4 = binary_probs4[0, 1].cpu().item()  # class 1의 확률
        prob_class5 = binary_probs5[0, 1].cpu().item()  # class 1의 확률        
        test_epoch_probs.append( (prob_class1+prob_class2+prob_class3+prob_class4+prob_class5)/5 )  # class 1 확률만 저장 (AUC용)
        target = c.cpu().item()
        test_epoch_targets.append(target)

test_epoch_probs = np.array(test_epoch_probs)
auc_score = roc_auc_score(test_epoch_targets, test_epoch_probs)

100%|██████████| 227/227 [00:31<00:00,  7.10it/s]


In [ ]:
df_risk_score_test = pd.DataFrame({"Patient_no":test_names, "risk_scores":test_epoch_probs, "labels": test_labels})
df_risk_score_test.to_csv("TEST_RISK_SCORES_FINAL_PNU_1patch_19slide.csv") 